# GOOGL Stock Movement Prediction
## Part 2: Feature Engineering

This notebook reads the three raw datasets collected in Part 1 and produces
a single aligned feature matrix for modeling.

**Design decisions:**
- Training window starts **2013-01-01** (one full year after first EDGAR
  filing at 2012-12-31, ensuring all QoQ and rolling fundamental features
  have sufficient lookback before the first training sample)
- Fundamental columns with >50% missing in the raw quarterly data are kept
  alongside a boolean availability mask (`{col}_available`), allowing the
  model to learn when those features are reliable
- `gross_profit` is derived as `revenue - cost_of_revenue` since Alphabet
  does not report it as a separate XBRL tag
- Macro series with monthly/quarterly native frequency are forward-filled
  without a day limit (a monthly release is valid until the next one arrives)
- Daily macro series use `ffill limit=5` to avoid propagating true gaps

**Output feature groups:**

| Group | Count |
|---|---|
| OHLCV (reference) | 5 |
| Technical | ~70 |
| Fundamental | ~47 |
| Availability masks | 3 |
| Macro | ~26 |
| Valuation | 6 |
| Targets | 3 |

In [3]:
!pip install pandas numpy pandas-ta scikit-learn -q

In [4]:
import os
import json
import warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:.6f}".format)

RAW_DIR       = "data/raw"
PROCESSED_DIR = "data/processed"
os.makedirs(PROCESSED_DIR, exist_ok=True)

# Training starts one year after first EDGAR filing (2012-12-31)
# to ensure QoQ and rolling fundamental features have valid lookback
TRAIN_START = "2014-01-01"
VAL_START   = "2022-01-01"
TEST_START  = "2024-01-01"

# Macro series that are monthly or quarterly — forward-fill without day limit
MACRO_LOW_FREQ = {
    "fed_funds_rate", "unemployment_rate", "cpi", "real_gdp",
    "cpi_mom", "gdp_qoq",
}

print(f"Training window : {TRAIN_START} to {VAL_START}")
print(f"Validation      : {VAL_START} to {TEST_START}")
print(f"Test            : {TEST_START} to present")

Training window : 2014-01-01 to 2022-01-01
Validation      : 2022-01-01 to 2024-01-01
Test            : 2024-01-01 to present


## Load Raw Data

In [5]:
price_raw = pd.read_csv(
    f"{RAW_DIR}/price_raw.csv",
    index_col="date",
    parse_dates=True,
)
# yfinance sometimes outputs a residual "Price" header row — drop if present
price_raw = price_raw[~price_raw.index.astype(str).str.contains("Price", na=False)]
price_raw = price_raw.apply(pd.to_numeric, errors="coerce")
price_raw.columns = [c.lower() for c in price_raw.columns]

fundamental_raw = pd.read_csv(
    f"{RAW_DIR}/fundamental_raw.csv",
    index_col="period_end",
    parse_dates=True,
)

macro_raw = pd.read_csv(
    f"{RAW_DIR}/macro_raw.csv",
    index_col="date",
    parse_dates=True,
)

print(f"Price       : {price_raw.shape}  |  {price_raw.index.min().date()} to {price_raw.index.max().date()}")
print(f"Fundamental : {fundamental_raw.shape}  |  {fundamental_raw.index.min().date()} to {fundamental_raw.index.max().date()}")
print(f"Macro       : {macro_raw.shape}  |  {macro_raw.index.min().date()} to {macro_raw.index.max().date()}")

Price       : (4140, 5)  |  2010-01-04 to 2026-06-18
Fundamental : (49, 29)  |  2012-12-31 to 2026-03-31
Macro       : (4351, 12)  |  2010-01-01 to 2026-06-18


## Alignment to Daily Trading Calendar
All datasets are reindexed to the price data's trading day index.
Forward-fill propagates the last known value forward — the standard approach
for mixed-frequency financial data.

**Forward-fill strategy by series type:**
- Fundamental (quarterly): no day limit — gaps up to ~92 days are normal
- Macro monthly/quarterly: no day limit — valid until next release
- Macro daily: limit=5 — longer gaps likely indicate true missing data

In [6]:
trading_index = price_raw.index

def align_fundamental(df: pd.DataFrame) -> pd.DataFrame:
    return df.reindex(trading_index, method=None).ffill(limit=None)

def align_macro(df: pd.DataFrame) -> pd.DataFrame:
    aligned = df.reindex(trading_index, method=None)
    for col in aligned.columns:
        if col in MACRO_LOW_FREQ:
            aligned[col] = aligned[col].ffill(limit=None)
        else:
            aligned[col] = aligned[col].ffill(limit=5)
    return aligned

fund_aligned  = align_fundamental(fundamental_raw)
macro_aligned = align_macro(macro_raw)

# S&P 500 from FRED has ~25% gaps. Replace with ^GSPC from yfinance
# which has complete daily coverage back to 2010.
import yfinance as yf
gspc = yf.download("^GSPC", start="2010-01-01", auto_adjust=True, progress=False)
if isinstance(gspc.columns, pd.MultiIndex):
    gspc.columns = gspc.columns.get_level_values(0)
gspc_close = gspc["Close"].rename("sp500")
gspc_close.index = pd.to_datetime(gspc_close.index)
macro_aligned["sp500"] = gspc_close.reindex(trading_index).ffill(limit=5)

print("After alignment:")
print(f"  Fund missing  : {fund_aligned.isnull().sum().sum():,} cells")
print(f"  Macro missing : {macro_aligned.isnull().sum().sum():,} cells")

# Remaining macro missing after this approach
macro_mv = macro_aligned.isnull().mean().sort_values(ascending=False)
remaining = macro_mv[macro_mv > 0]
if not remaining.empty:
    print("\nRemaining macro missing (%) after ffill:")
    print(remaining.to_string())

After alignment:
  Fund missing  : 33,932 cells
  Macro missing : 118 cells

Remaining macro missing (%) after ffill:
real_gdp            0.014734
fed_funds_rate      0.004589
unemployment_rate   0.004589
cpi                 0.004589


## Availability Masks for Sparse Fundamental Columns

Columns with >50% missing in the raw quarterly data are kept but supplemented
with a binary mask. The mask is `1` on days where the company actually
reported that value in a filing, and `0` on days where the forward-filled
value is stale beyond one quarter. This lets the model weight sparse features
appropriately rather than treating stale forward-filled values as fresh data.

In [7]:
SPARSE_THRESHOLD = 0.50

missing_pct = fundamental_raw.isnull().mean()
sparse_cols = missing_pct[missing_pct > SPARSE_THRESHOLD].index.tolist()
print(f"Sparse fundamental columns (>{SPARSE_THRESHOLD*100:.0f}% missing in raw): {sparse_cols}")

mask_frames = {}
for col in sparse_cols:
    reported_dates = fundamental_raw.index[fundamental_raw[col].notna()]
    mask = pd.Series(0, index=trading_index, name=f"{col}_available", dtype=int)
    mask[mask.index.isin(reported_dates)] = 1
    mask_frames[col] = mask

availability_masks = pd.DataFrame(mask_frames)
print(f"\nDays with a reported value per sparse column:")
print(availability_masks.sum().to_string())

Sparse fundamental columns (>50% missing in raw): ['dda', 'short_term_investments', 'long_term_debt']

Days with a reported value per sparse column:
dda                       10
short_term_investments     8
long_term_debt             7


## Technical Feature Engineering

All technical features are computed solely from price and volume data.
No future data is used — indicators that require a warmup period produce
NaN at the start, which are handled when the feature matrix is trimmed to
TRAIN_START at assembly time.

**Feature groups:**
1. Log returns at multiple lookback horizons
2. SMA and EMA with price-relative normalization
3. Momentum: MACD, RSI, Stochastic, Williams %R, CCI, ADX
4. Volatility: Bollinger Bands, ATR, realized volatility
5. Volume: OBV, VWAP, MFI, volume ratios
6. Range/channel: Donchian, Keltner, Ichimoku Tenkan/Kijun, candlestick wicks

In [8]:
import pandas_ta as ta

close  = price_raw["close"]
high   = price_raw["high"]
low    = price_raw["low"]
open_  = price_raw["open"]
volume = price_raw["volume"]

tech = pd.DataFrame(index=trading_index)

# Log returns at multiple horizons
for n in [1, 2, 3, 5, 10, 20, 60]:
    tech[f"log_return_{n}d"] = np.log(close / close.shift(n))

# SMA and price position relative to SMA
for n in [5, 10, 20, 50, 200]:
    sma = close.rolling(n).mean()
    tech[f"sma_{n}"]         = sma
    tech[f"price_to_sma_{n}"] = close / sma - 1

# EMA and price position relative to EMA
for n in [5, 10, 20, 50]:
    ema = close.ewm(span=n, adjust=False).mean()
    tech[f"ema_{n}"]         = ema
    tech[f"price_to_ema_{n}"] = close / ema - 1

print(f"Returns and MA features: {tech.shape[1]}")

Returns and MA features: 25


In [9]:
# MACD (12, 26, 9)
ema12 = close.ewm(span=12, adjust=False).mean()
ema26 = close.ewm(span=26, adjust=False).mean()
tech["macd_line"]   = ema12 - ema26
tech["macd_signal"] = tech["macd_line"].ewm(span=9, adjust=False).mean()
tech["macd_hist"]   = tech["macd_line"] - tech["macd_signal"]

# RSI (14)
delta = close.diff()
gain  = delta.clip(lower=0).rolling(14).mean()
loss  = (-delta.clip(upper=0)).rolling(14).mean()
tech["rsi_14"] = 100 - (100 / (1 + gain / loss.replace(0, np.nan)))

# Stochastic %K and %D (14, 3)
low14   = low.rolling(14).min()
high14  = high.rolling(14).max()
stoch_k = 100 * (close - low14) / (high14 - low14).replace(0, np.nan)
tech["stoch_k"] = stoch_k
tech["stoch_d"] = stoch_k.rolling(3).mean()

# Williams %R (14)
tech["williams_r"] = -100 * (high14 - close) / (high14 - low14).replace(0, np.nan)

# CCI (20)
typical   = (high + low + close) / 3
sma_tp20  = typical.rolling(20).mean()
mad20     = typical.rolling(20).apply(lambda x: np.abs(x - x.mean()).mean(), raw=True)
tech["cci_20"] = (typical - sma_tp20) / (0.015 * mad20.replace(0, np.nan))

# ADX (14) via pandas-ta — avoids manual edge case handling
adx_df         = ta.adx(high, low, close, length=14)
tech["adx_14"] = adx_df["ADX_14"]
tech["dmp_14"] = adx_df["DMP_14"]
tech["dmn_14"] = adx_df["DMN_14"]

print(f"Momentum features added. Total: {tech.shape[1]}")

Momentum features added. Total: 36


In [10]:
# Bollinger Bands (20, ±2σ)
sma20    = close.rolling(20).mean()
std20    = close.rolling(20).std()
bb_upper = sma20 + 2 * std20
bb_lower = sma20 - 2 * std20
tech["bb_upper"]    = bb_upper
tech["bb_lower"]    = bb_lower
tech["bb_width"]    = (bb_upper - bb_lower) / sma20
tech["bb_position"] = (close - bb_lower) / (bb_upper - bb_lower).replace(0, np.nan)

# ATR (14) — exponential smoothing of true range
tr = pd.concat([
    high - low,
    (high - close.shift(1)).abs(),
    (low  - close.shift(1)).abs(),
], axis=1).max(axis=1)
tech["atr_14"]     = tr.ewm(span=14, adjust=False).mean()
tech["atr_14_pct"] = tech["atr_14"] / close

# Realized volatility at multiple windows (annualized)
log_ret = np.log(close / close.shift(1))
for n in [10, 20, 60]:
    tech[f"hvol_{n}d"] = log_ret.rolling(n).std() * np.sqrt(252)

print(f"Volatility features added. Total: {tech.shape[1]}")

Volatility features added. Total: 45


In [11]:
# On-Balance Volume
obv = (np.sign(close.diff()) * volume).fillna(0).cumsum()
tech["obv"]          = obv
tech["obv_sma20"]    = obv.rolling(20).mean()
tech["obv_momentum"] = obv / obv.rolling(20).mean() - 1

# Volume ratios
tech["volume_ratio_5_20"]  = volume.rolling(5).mean()  / volume.rolling(20).mean()
tech["volume_ratio_20_60"] = volume.rolling(20).mean() / volume.rolling(60).mean()

# Volume z-score
vol_mean20 = volume.rolling(20).mean()
vol_std20  = volume.rolling(20).std()
tech["volume_zscore_20"] = (volume - vol_mean20) / vol_std20.replace(0, np.nan)

# VWAP (20-day rolling approximation using daily bars)
typical_price     = (high + low + close) / 3
tech["vwap_20d"]  = (typical_price * volume).rolling(20).sum() / volume.rolling(20).sum()
tech["price_to_vwap"] = close / tech["vwap_20d"] - 1

# Money Flow Index (14)
raw_mf = typical_price * volume
pos_mf = raw_mf.where(typical_price > typical_price.shift(1), 0)
neg_mf = raw_mf.where(typical_price < typical_price.shift(1), 0)
mfr    = pos_mf.rolling(14).sum() / neg_mf.rolling(14).sum().replace(0, np.nan)
tech["mfi_14"] = 100 - (100 / (1 + mfr))

print(f"Volume features added. Total: {tech.shape[1]}")

Volume features added. Total: 54


In [12]:
# Donchian Channel (20)
dc_upper = high.rolling(20).max()
dc_lower = low.rolling(20).min()
tech["donchian_width"]    = (dc_upper - dc_lower) / close
tech["donchian_position"] = (close - dc_lower) / (dc_upper - dc_lower).replace(0, np.nan)

# Keltner Channel (20, ATR multiplier=2)
kc_mid   = close.ewm(span=20, adjust=False).mean()
kc_upper = kc_mid + 2 * tech["atr_14"]
kc_lower = kc_mid - 2 * tech["atr_14"]
tech["keltner_position"] = (close - kc_lower) / (kc_upper - kc_lower).replace(0, np.nan)

# Candlestick wick features
body        = (close - open_).abs()
total_range = (high - low).replace(0, np.nan)
tech["wick_upper_ratio"] = (high - pd.concat([close, open_], axis=1).max(axis=1)) / total_range
tech["wick_lower_ratio"] = (pd.concat([close, open_], axis=1).min(axis=1) - low) / total_range
tech["body_ratio"]       = body / total_range
tech["candle_direction"] = np.sign(close - open_)

# Ichimoku Tenkan / Kijun
tenkan = (high.rolling(9).max()  + low.rolling(9).min())  / 2
kijun  = (high.rolling(26).max() + low.rolling(26).min()) / 2
tech["tenkan_kijun_spread"] = (tenkan - kijun) / close
tech["price_to_kijun"]      = close / kijun - 1

# Rolling 60-day correlation with NASDAQ — filled after macro section
tech["price_corr_nasdaq_60d"] = np.nan

print(f"Channel and range features added. Total: {tech.shape[1]}")

Channel and range features added. Total: 64


## Fundamental Feature Engineering

Ratios are computed from the forward-filled quarterly data and are therefore
available at daily frequency. All denominators use `.replace(0, np.nan)` to
avoid division by zero producing inf values.

In [13]:
f = fund_aligned.copy()

# Gross profit not reported as XBRL tag by Alphabet — derive it
f["gross_profit"] = f["revenue"] - f["cost_of_revenue"]

fund = pd.DataFrame(index=trading_index)

# Profitability margins
eps_revenue = f["revenue"].replace(0, np.nan)
fund["gross_margin"]      = f["gross_profit"]     / eps_revenue
fund["operating_margin"]  = f["operating_income"] / eps_revenue
fund["net_margin"]        = f["net_income"]        / eps_revenue
fund["rd_ratio"]          = f["rd_expense"]        / eps_revenue
fund["sga_ratio"]         = f["sga_expense"]       / eps_revenue
fund["cost_ratio"]        = f["cost_of_revenue"]   / eps_revenue
fund["tax_rate"]          = f["income_tax"] / f["operating_income"].replace(0, np.nan)

# Return metrics
avg_equity = f["stockholders_equity"].rolling(2).mean().replace(0, np.nan)
avg_assets = f["total_assets"].rolling(2).mean().replace(0, np.nan)
fund["roe"] = f["net_income"] / avg_equity
fund["roa"] = f["net_income"] / avg_assets

# ROIC = NOPAT / Invested Capital
nopat            = f["operating_income"] * (1 - fund["tax_rate"].clip(0, 0.5))
invested_capital = (f["total_assets"] - f["current_liabilities"] - f["cash"]).replace(0, np.nan)
fund["roic"]     = nopat / invested_capital

# Liquidity
fund["current_ratio"] = f["current_assets"] / f["current_liabilities"].replace(0, np.nan)
fund["quick_ratio"]   = fund["current_ratio"]  # no inventory for Alphabet

# Leverage
fund["debt_to_equity"] = f["long_term_debt"]     / f["stockholders_equity"].replace(0, np.nan)
fund["debt_to_assets"] = f["total_liabilities"]  / f["total_assets"].replace(0, np.nan)
fund["equity_ratio"]   = f["stockholders_equity"] / f["total_assets"].replace(0, np.nan)

# Efficiency
fund["asset_turnover"]         = f["revenue"] / avg_assets
fund["receivables_turnover"]   = f["revenue"] / f["accounts_receivable"].replace(0, np.nan)
fund["days_sales_outstanding"] = 365 / fund["receivables_turnover"].replace(0, np.nan)

# Cash flow quality
fund["ocf_margin"]       = f["operating_cash_flow"] / eps_revenue
fund["fcf_margin"]       = (f["operating_cash_flow"] - f["capex"]) / eps_revenue
fund["capex_intensity"]  = f["capex"].abs() / f["total_assets"].replace(0, np.nan)
fund["ocf_to_netincome"] = f["operating_cash_flow"] / f["net_income"].replace(0, np.nan)

# Accruals ratio (earnings quality — lower is better)
avg_net_assets    = (f["total_assets"] - f["total_liabilities"]).rolling(2).mean().replace(0, np.nan)
fund["accruals_ratio"] = (f["net_income"] - f["operating_cash_flow"]) / avg_net_assets

# Intangible and innovation intensity
fund["rd_to_assets"]     = f["rd_expense"]  / f["total_assets"].replace(0, np.nan)
fund["goodwill_ratio"]   = f["goodwill"]    / f["total_assets"].replace(0, np.nan)
fund["intangible_ratio"] = f["intangibles"] / f["total_assets"].replace(0, np.nan)

print(f"Base fundamental ratios: {fund.shape[1]}")

Base fundamental ratios: 26


In [14]:
# 9 binary signals summed to 0-9. Each component is also kept individually
# since it may carry signal the aggregate score obscures.

pf = pd.DataFrame(index=trading_index)

pf["pf_roa_pos"]       = (fund["roa"] > 0).astype(int)
pf["pf_ocf_pos"]       = (f["operating_cash_flow"] > 0).astype(int)
pf["pf_roa_improving"] = (fund["roa"] > fund["roa"].shift(252)).astype(int)
pf["pf_accruals_low"]  = (fund["accruals_ratio"] < 0).astype(int)

pf["pf_leverage_dec"]  = (fund["debt_to_assets"] < fund["debt_to_assets"].shift(252)).astype(int)
pf["pf_liquidity_inc"] = (fund["current_ratio"]  > fund["current_ratio"].shift(252)).astype(int)
pf["pf_no_dilution"]   = (f["shares_outstanding"] <= f["shares_outstanding"].shift(252)).astype(int)

pf["pf_margin_inc"]    = (fund["gross_margin"]   > fund["gross_margin"].shift(252)).astype(int)
pf["pf_turnover_inc"]  = (fund["asset_turnover"] > fund["asset_turnover"].shift(252)).astype(int)

pf["piotroski_f_score"] = pf.sum(axis=1)

fund = pd.concat([fund, pf], axis=1)
print(f"Piotroski F-Score added. Total fundamental features: {fund.shape[1]}")

Piotroski F-Score added. Total fundamental features: 36


In [15]:
# Z' = 6.56*X1 + 3.26*X2 + 6.72*X3 + 1.05*X4

az = pd.DataFrame(index=trading_index)

total_assets    = f["total_assets"].replace(0, np.nan)
working_capital = f["current_assets"] - f["current_liabilities"]

az["az_x1"] = working_capital          / total_assets
az["az_x2"] = f["retained_earnings"]   / total_assets
az["az_x3"] = f["operating_income"]    / total_assets
az["az_x4"] = f["stockholders_equity"] / f["total_liabilities"].replace(0, np.nan)

az["altman_z_score"] = (
    6.56 * az["az_x1"] +
    3.26 * az["az_x2"] +
    6.72 * az["az_x3"] +
    1.05 * az["az_x4"]
)

fund = pd.concat([fund, az], axis=1)
print(f"Altman Z-Score added. Total fundamental features: {fund.shape[1]}")

Altman Z-Score added. Total fundamental features: 41


In [16]:
# 63 trading days ≈ 1 quarter on the daily index

QOQ_SHIFT = 63

qoq_map = {
    "revenue"            : "revenue_qoq",
    "net_income"         : "net_income_qoq",
    "operating_income"   : "op_income_qoq",
    "operating_cash_flow": "ocf_qoq",
    "rd_expense"         : "rd_qoq",
    "eps_diluted"        : "eps_qoq",
}

for src_col, out_col in qoq_map.items():
    prev  = f[src_col].shift(QOQ_SHIFT)
    fund[out_col] = (f[src_col] - prev) / prev.abs().replace(0, np.nan)

print(f"QoQ growth features added. Total fundamental features: {fund.shape[1]}")

QoQ growth features added. Total fundamental features: 47


## Macro Feature Engineering

In [17]:
macro = macro_aligned.copy()

# Yield curve spread (negative = inverted, historically precedes recessions)
macro["yield_spread_10_2"]    = macro["treasury_10y"] - macro["treasury_2y"]

# Real 10-year yield
macro["real_rate_10y"]        = macro["treasury_10y"] - macro["breakeven_inflation_10y"]

# VIX regime: 0=low (<20), 1=medium (20-30), 2=high (>30)
macro["vix_regime"] = pd.cut(
    macro["vix"], bins=[-np.inf, 20, 30, np.inf], labels=[0, 1, 2]
).astype(float)

# Rate of change for daily macro series
for col in ["vix", "treasury_10y", "oil_price_wti", "usd_index"]:
    macro[f"{col}_5d_chg"]  = macro[col].pct_change(5)
    macro[f"{col}_20d_chg"] = macro[col].pct_change(20)

# Monthly series changes
macro["cpi_mom"] = macro["cpi"].pct_change()
macro["gdp_qoq"] = macro["real_gdp"].pct_change()

# GOOGL rolling 60-day correlation with NASDAQ
nasdaq_ret = macro["nasdaq_composite"].pct_change()
price_ret  = close.pct_change()
tech["price_corr_nasdaq_60d"] = price_ret.rolling(60).corr(nasdaq_ret)

# GOOGL return relative to NASDAQ over 20 days (alpha proxy)
macro["googl_vs_nasdaq_20d"] = (
    np.log(close / close.shift(20)) -
    np.log(macro["nasdaq_composite"] / macro["nasdaq_composite"].shift(20))
)

print(f"Macro features: {macro.shape[1]}")

Macro features: 26


## Valuation Features (Price × Fundamental)

In [18]:
val = pd.DataFrame(index=trading_index)

market_cap = close * f["shares_outstanding"]

# P/E — diluted EPS annualized (×4 for quarterly filings)
eps_annual      = f["eps_diluted"] * 4
val["pe_ratio"] = close / eps_annual.replace(0, np.nan)

# P/B
book_per_share  = f["stockholders_equity"] / f["shares_outstanding"].replace(0, np.nan)
val["pb_ratio"] = close / book_per_share.replace(0, np.nan)

# P/S
sales_per_share = f["revenue"] / f["shares_outstanding"].replace(0, np.nan)
val["ps_ratio"] = close / sales_per_share.replace(0, np.nan)

# EV/EBITDA — long_term_debt is sparse; fillna(0) treats missing as no debt
net_debt          = f["long_term_debt"].fillna(0) - f["cash"].fillna(0)
ev                = market_cap + net_debt
ebitda            = f["operating_income"] + f["dda"].fillna(0)
val["ev_ebitda"]  = ev / ebitda.replace(0, np.nan)

# FCF yield
fcf               = f["operating_cash_flow"] - f["capex"].abs()
val["fcf_yield"]  = fcf / market_cap.replace(0, np.nan)

# Log market cap (size feature)
val["log_market_cap"] = np.log(market_cap.replace(0, np.nan))

print(f"Valuation features: {val.shape[1]}")

Valuation features: 6


## Target Variables

Three forward return targets at 1, 5, and 20 trading day horizons.
The final `n` rows will have NaN targets (no future price available) and
are excluded from training/validation but kept for inference.

In [19]:
targets = pd.DataFrame(index=trading_index)

for n in [1, 5, 20]:
    targets[f"target_return_{n}d"] = close.shift(-n) / close - 1

print("Target variable statistics (full history):")
print(targets.describe().to_string())
print()
for col in targets.columns:
    pos = (targets[col] > 0).mean()
    print(f"  {col}: {pos:.1%} positive days")

Target variable statistics (full history):
       target_return_1d  target_return_5d  target_return_20d
count       4139.000000       4135.000000        4120.000000
mean           0.000917          0.004576           0.018601
std            0.017484          0.037979           0.073586
min           -0.116341         -0.154561          -0.293489
25%           -0.007578         -0.017209          -0.028532
50%            0.000882          0.005263           0.017696
75%            0.009536          0.025263           0.060961
max            0.162584          0.258060           0.304020

  target_return_1d: 52.9% positive days
  target_return_5d: 56.4% positive days
  target_return_20d: 60.5% positive days


## Assemble Final Feature Matrix

In [20]:
df = pd.concat([
    price_raw,
    tech,
    fund,
    fund_aligned[sparse_cols].add_prefix("raw_"),
    availability_masks,
    macro,
    val,
    targets,
], axis=1)

# Trim to training start — drops the pre-2013 warmup period
df = df[df.index >= TRAIN_START]

# Clip extreme values at 1st/99th percentile per feature
# Excludes raw OHLCV, binary/categorical columns, and targets
exclude_from_clip = (
    list(price_raw.columns) +
    list(targets.columns) +
    list(availability_masks.columns) +
    ["candle_direction", "vix_regime", "piotroski_f_score"] +
    [c for c in pf.columns]
)
clip_cols = [c for c in df.columns if c not in exclude_from_clip]
lower = df[clip_cols].quantile(0.01)
upper = df[clip_cols].quantile(0.99)
df[clip_cols] = df[clip_cols].clip(lower=lower, upper=upper, axis=1)

print(f"Final feature matrix shape : {df.shape}")
print(f"Date range                 : {df.index.min().date()} to {df.index.max().date()}")
print()

missing_pct = df.isnull().mean().sort_values(ascending=False)
high_missing = missing_pct[missing_pct > 0.10]
if not high_missing.empty:
    print(f"Columns with >10% missing ({len(high_missing)}):")
    print(high_missing.to_string())
else:
    print("No columns with >10% missing.")

Final feature matrix shape : (3134, 157)
Date range                 : 2014-01-02 to 2026-06-18

Columns with >10% missing (7):
raw_dda              0.642629
accruals_ratio       0.160817
debt_to_equity       0.160498
debt_to_assets       0.160498
raw_long_term_debt   0.160498
altman_z_score       0.160498
az_x4                0.160498


In [21]:
split_index = {
    "train_start" : TRAIN_START,
    "train_end"   : (pd.Timestamp(VAL_START)  - pd.Timedelta(days=1)).strftime("%Y-%m-%d"),
    "val_start"   : VAL_START,
    "val_end"     : (pd.Timestamp(TEST_START) - pd.Timedelta(days=1)).strftime("%Y-%m-%d"),
    "test_start"  : TEST_START,
    "test_end"    : df.index.max().strftime("%Y-%m-%d"),
}

valid_rows = df[df["target_return_1d"].notna()]
split_index["train_rows"] = int((valid_rows.index <  VAL_START).sum())
split_index["val_rows"]   = int(((valid_rows.index >= VAL_START) & (valid_rows.index < TEST_START)).sum())
split_index["test_rows"]  = int((valid_rows.index >= TEST_START).sum())

print("Walk-forward split:")
for k, v in split_index.items():
    print(f"  {k:<15}: {v}")

with open(f"{PROCESSED_DIR}/split_index.json", "w") as fh:
    json.dump(split_index, fh, indent=2)

Walk-forward split:
  train_start    : 2014-01-01
  train_end      : 2021-12-31
  val_start      : 2022-01-01
  val_end        : 2023-12-31
  test_start     : 2024-01-01
  test_end       : 2026-06-18
  train_rows     : 2015
  val_rows       : 501
  test_rows      : 617


In [23]:
feature_groups = {
    "ohlcv"                  : list(price_raw.columns),
    "technical"              : list(tech.columns),
    "fundamental"            : list(fund.columns),
    "fundamental_sparse_raw" : [f"raw_{c}" for c in sparse_cols],
    "availability_masks"     : list(availability_masks.columns),
    "macro"                  : list(macro.columns),
    "valuation"              : list(val.columns),
    "targets"                : list(targets.columns),
}

all_cataloged   = sum(feature_groups.values(), [])
missing_in_df   = [c for c in all_cataloged if c not in df.columns]
if missing_in_df:
    print(f"Warning: {len(missing_in_df)} cataloged columns missing from df: {missing_in_df}")

with open(f"{PROCESSED_DIR}/feature_groups.json", "w") as fh:
    json.dump(feature_groups, fh, indent=2)

df.to_csv(f"{PROCESSED_DIR}/features.csv")

print("  FEATURE ENGINEERING SUMMARY")
for group, cols in feature_groups.items():
    print(f"  {group:<28}: {len(cols):>3} columns")
print(f"  {'TOTAL':<28}: {len(all_cataloged):>3} columns")
print()
print(f"  features.csv        : {PROCESSED_DIR}/features.csv")
print(f"  split_index.json    : {PROCESSED_DIR}/split_index.json")
print(f"  feature_groups.json : {PROCESSED_DIR}/feature_groups.json")
print()
print("  Ready for Part 3: EDA")

  FEATURE ENGINEERING SUMMARY
  ohlcv                       :   5 columns
  technical                   :  64 columns
  fundamental                 :  47 columns
  fundamental_sparse_raw      :   3 columns
  availability_masks          :   3 columns
  macro                       :  26 columns
  valuation                   :   6 columns
  targets                     :   3 columns
  TOTAL                       : 157 columns

  features.csv        : data/processed/features.csv
  split_index.json    : data/processed/split_index.json
  feature_groups.json : data/processed/feature_groups.json

  Ready for Part 3: EDA
